# Lab 2 — Lakeflow Jobs concepts (Course 2)
Lessons 2.1–2.6 · exercises `2a`–`2d` · schema `lab2`

Real jobs are built in the **Jobs UI** (or API / SDK / bundles). Here `labutils.jobsim.Job` runs Python
functions as tasks with the **same rules**: DAG dependencies, run-if, If/else, For each, parameters,
task values, `{{dynamic references}}`, retries and repair. Do the real UI version in Databricks Free Edition later.

| Databricks | This lab |
|---|---|
| Notebook task | `@job.task("key")` function — gets `ctx` |
| `dbutils.widgets.get("x")` | `ctx.widget("x")` |
| `dbutils.jobs.taskValues.set(key=, value=)` | `ctx.task_values.set(key, value)` |
| `dbutils.jobs.taskValues.get(taskKey=, key=)` | `ctx.task_values.get(taskKey, key)` |
| If/else condition task | `job.if_else(...)` |
| For each task | `job.for_each(...)` — item is `ctx.input` / `{{input}}` |
| Repair run | `job.repair(run)` |

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
from labutils.jobsim import Job
reset_lab("lab2")

## 2a · A multi-task job is a DAG (lessons 2.1–2.2)
Two ingestion tasks have no dependency → they can run **in parallel**. The join task must wait for **both**
(a *funnel* pattern). **Task:** set `depends_on` for the join task.

In [ ]:
# TODO 2a
job = Job("retail_job")

@job.task("ingest_orders")
def ingest_orders(ctx):
    landing.reset("orders"); landing.land("orders", 2)
    spark.read.json(landing.volume("orders")).write.mode("overwrite").saveAsTable("lab2.orders_bronze")

@job.task("ingest_customers")
def ingest_customers(ctx):
    (spark.range(1, 41).selectExpr("id AS customer_id", "concat('customer_', id) AS name")
          .write.mode("overwrite").saveAsTable("lab2.customers"))

@job.task("join_customers_orders", depends_on=<FILL-IN 1>)
def join_customers_orders(ctx):
    spark.sql("""CREATE OR REPLACE TABLE lab2.customers_orders AS
                 SELECT o.*, c.name FROM lab2.orders_bronze o JOIN lab2.customers c USING (customer_id)""")

run = job.run()
run.save("2a")

In [ ]:
check("2a")
# Why "Directed ACYCLIC Graph"? Try a cycle:
from labutils.jobsim import Task
bad = Job("cycle")
bad.add_task(Task("a", lambda c: None, ["b"]))
bad.add_task(Task("b", lambda c: None, ["a"]))
try:
    bad.run()
except ValueError as e:
    print("ERROR:", e)

## 2b · Parameters and task values (lesson 2.3)
- **Task parameters** belong to one task. **Job parameters** apply to every task and **override** a task
  parameter with the same key. Both can be overridden at run time (*Run now with different parameters*).
- **Task values** pass results from one task to a later task.
- **Dynamic value references**: `{{tasks.<task_key>.values.<key>}}`, `{{job.parameters.<name>}}`, `{{job.run_id}}` …

**Task:** (1) save the row count as task value `row_count`; (2) pass it to `report` with a dynamic reference;
(3) run the job with job parameter `catalog = "lab2_prod"`. Predict first: which `catalog` does `count_orders` print?

In [ ]:
# TODO 2b
job = Job("params_job", parameters={"catalog": "lab2_dev"})

@job.task("count_orders", parameters={"catalog": "task_default", "min_id": "75000"})
def count_orders(ctx):
    print("count_orders sees catalog =", ctx.widget("catalog"))
    n = spark.table("lab2.orders_bronze").filter(f"order_id > {ctx.widget('min_id')}").count()
    <FILL-IN 1>                                   # share n as task value "row_count"

@job.task("report", depends_on=["count_orders"], parameters={"rows": "<FILL-IN 2>"})
def report(ctx):
    print(f"report: catalog={ctx.widget('catalog')} rows={ctx.widget('rows')}")
    print("same value read directly:", ctx.task_values.get(taskKey="count_orders", key="row_count"))

run = job.run(parameters=<FILL-IN 3>)
run.save("2b")

In [ ]:
check("2b")

## 2c · Run-if, If/else and For each (lesson 2.5)
- **If/else task** compares two values (`==, !=, >, >=, <, <=`) → downstream tasks hang on the **True** or **False** branch.
  The branch not taken is **Excluded**.
- **For each task** runs one nested task per item of an array; the item is `{{input}}` (here `ctx.input`).
- **Run if**: `ALL_SUCCEEDED` (default), `AT_LEAST_ONE_SUCCEEDED`, `NONE_FAILED`, `ALL_DONE`, `AT_LEAST_ONE_FAILED`, `ALL_FAILED`.

**Task:** the For each task depends on **both** branches. One branch is always excluded, so which `run_if`
lets it run? (Try `ALL_SUCCEEDED` first and see what happens.)

In [ ]:
# TODO 2c
job = Job("advanced_job")

@job.task("check_duplicates")
def check_duplicates(ctx):
    df = spark.table("lab2.orders_bronze")
    has = df.count() != df.dropDuplicates(["order_id"]).count()
    ctx.task_values.set("has_duplicates", str(has).lower())

job.if_else("duplicates_exist", left="<FILL-IN 1>", op="==", right="true",
            depends_on=["check_duplicates"])

@job.task("drop_duplicates", depends_on=[("duplicates_exist", True)])
def drop_duplicates(ctx):
    spark.table("lab2.orders_bronze").dropDuplicates(["order_id"]).write.mode("overwrite").saveAsTable("lab2.orders_clean")

@job.task("no_duplicates", depends_on=[("duplicates_exist", False)])
def no_duplicates(ctx):
    spark.table("lab2.orders_bronze").write.mode("overwrite").saveAsTable("lab2.orders_clean")

STATES = ["CA", "NY", "VA"]
def state_report(ctx):                  # the nested task: runs once per state
    i = STATES.index(ctx.input)
    (spark.table("lab2.orders_clean").filter(f"customer_id % 3 = {i}")
          .write.mode("overwrite").saveAsTable(f"lab2.orders_{ctx.input.lower()}"))

job.for_each("state_reports", inputs=STATES, fn=state_report, concurrency=3,
             depends_on=["drop_duplicates", "no_duplicates"], run_if="<FILL-IN 2>")

run = job.run()
run.save("2c")

In [ ]:
check("2c")

## 2d · Retries and repair runs (lesson 2.6)
- **Retry policy**: a task can retry automatically (e.g. `max_retries=2` → up to 3 attempts).
- **Repair run**: after a failure, re-run **only the failed tasks and the tasks after them**. Successful tasks are
  **not** re-run (saves time and money). You may change parameters for the repair.

The job below has a bug: a wrong table name in a **parameter**. **Task:** repair the run with the right parameter.

In [ ]:
attempts = {"n": 0}
job = Job("flaky_job")

@job.task("ingest", max_retries=2)
def ingest(ctx):
    attempts["n"] += 1
    if attempts["n"] == 1:
        raise ConnectionError("temporary network problem")   # 1st attempt fails, retry succeeds
    print("ingest ok on attempt", attempts["n"])

@job.task("transform", depends_on=["ingest"], parameters={"source_table": "lab2.orders_bronz"})
def transform(ctx):
    n = spark.table(ctx.widget("source_table")).count()
    ctx.task_values.set("rows", n)

@job.task("publish", depends_on=["transform"])
def publish(ctx):
    print("published", ctx.task_values.get("transform", "rows"), "rows")

run = job.run()

In [ ]:
# TODO 2d: repair the failed run, overriding the parameter source_table with the correct table
run = job.repair(run, parameters=<FILL-IN 1>)
run.save("2d")

In [ ]:
check("2d")
print("ingest ran", attempts["n"], "times in total — the repair did NOT re-run it.")

## Next
- Build the same jobs in the real **Jobs UI**: course notebooks `1 Demo`…`6 Demo` in Databricks Free Edition.
- Triggers (schedule/cron, file arrival, table update, continuous) and notifications are UI-only — study `notes/c2-jobs.md` §4.
- Ask Claude: `/quiz jobs 5`.